# 07 · Tier 2 — Graph & multimodal

Examine **Tier 2** implementations:

1. **LightGCN** — BPR training on the user–item bipartite graph
2. **Image embeddings** — CLIP vectors + `ImageEmbeddingRecommender` (when `has_images=True`)
3. **Unified pipeline** — image retriever arm in `MultiRetriever`
4. **Optional ranker feature** — `use_lightgcn` in two-stage

Build image vectors: `python scripts/build_image_vectors.py` (Yelp photos in `data/raw/photos/`).

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.recsys.config import settings
from src.recsys.data import load_dataset
from src.recsys.models import (
    ImageEmbeddingRecommender,
    LightGCNRecommender,
    TwoStageRecommender,
    TwoTowerRecommender,
)
from scripts.benchmark import (
    _score_recs,
    build_slices,
    build_unified_two_stage,
    slice_ground_truth,
)

ds = load_dataset()
train, test_pos, cold_items, cold_users = build_slices(ds, cutoff_quantile=0.9)
warm_gt, ci_gt, cu_gt = slice_ground_truth(test_pos, cold_items, cold_users)
users = sorted(set(warm_gt) | set(ci_gt) | set(cu_gt))
k = settings.top_k

print(ds.summary())
if not ds.item_image_vectors:
    print("\nNote: no item_image_vectors.npy — image sections will be skipped.")
    print("Run: python scripts/build_image_vectors.py")

## 1. LightGCN (graph collaborative filtering)

Neighborhood aggregation on the interaction graph. Expect strong **warm**, near-zero cold slices.

In [ ]:
NB = dict(dim=64, n_layers=2, epochs=6)  # benchmark: 10 epochs

lightgcn = LightGCNRecommender(**NB)
lightgcn.fit(train)
lg_recs = lightgcn.recommend(users, k=k)
lg_metrics = _score_recs(lg_recs, test_pos, warm_gt, ci_gt, cu_gt, k)
pd.Series(lg_metrics, name="lightgcn")

## 2. Image embedding retriever

CLIP mean-pooled user profiles over item image vectors. Helps **cold-user** (popularity fallback when no profile).

In [ ]:
img_metrics = None
if ds.item_image_vectors is not None:
    image = ImageEmbeddingRecommender(
        items=ds.items, item_image_vectors=ds.item_image_vectors
    )
    image.fit(train)
    img_recs = image.recommend(users, k=k)
    img_metrics = _score_recs(img_recs, test_pos, warm_gt, ci_gt, cu_gt, k)
    display(pd.Series(img_metrics, name="image_embedding"))

    n_with_photos = int((ds.item_image_vectors.sum(axis=1) != 0).sum())
    print(f"Items with non-zero image vectors: {n_with_photos:,} / {ds.n_items:,}")
else:
    print("Skipped — build vectors first.")

## 3. Unified pipeline with image arm

When image vectors exist, `build_unified_two_stage` auto-inserts `ImageEmbeddingRecommender` into the retriever fusion.

In [ ]:
unified = build_unified_two_stage(ds, use_sasrec=False)
print("Retriever arms:", [r.name for r in unified.retriever.retrievers])

unified.fit(train)
uni_recs = unified.recommend(users, k=k)
uni_metrics = _score_recs(uni_recs, test_pos, warm_gt, ci_gt, cu_gt, k)
pd.Series(uni_metrics, name="two_stage_unified")

## 4. LightGCN as ranker feature (`use_lightgcn`)

Same pattern as SASRec: fit graph model on train, score candidates only — no extra retriever arm.

In [ ]:
from src.recsys.models import MultiRetriever, ContentBasedRecommender, PopularityRecommender, SocialRecommender

def _make_retriever():
    return MultiRetriever([
        TwoTowerRecommender(dim=64, epochs=6),
        ContentBasedRecommender(items=ds.items),
        SocialRecommender(social=ds.social),
        PopularityRecommender(),
    ])

with_lgc = TwoStageRecommender(
    _make_retriever(),
    candidate_n=200,
    use_social=True,
    social=ds.social,
    use_lightgcn=True,
    lightgcn_kwargs=NB,
    use_extended_features=True,
)
without_lgc = TwoStageRecommender(
    _make_retriever(),
    candidate_n=200,
    use_social=True,
    social=ds.social,
    use_lightgcn=False,
    use_extended_features=True,
)

ab_rows = []
for name, model in [("unified", without_lgc), ("unified+lightgcn", with_lgc)]:
    model.fit(train)
    recs = model.recommend(users, k=k)
    ab_rows.append({"model": name, **_score_recs(recs, test_pos, warm_gt, ci_gt, cu_gt, k)})

pd.DataFrame(ab_rows).set_index("model")[[f"overall_r@{k}", f"warm_r@{k}", f"cold_item_r@{k}", f"cold_user_r@{k}"]]

## 5. Tier 2 comparison table

In [ ]:
rows = [{"model": "lightgcn", **lg_metrics}, {"model": "two_stage_unified", **uni_metrics}]
if img_metrics:
    rows.insert(1, {"model": "image_embedding", **img_metrics})

tier2_df = pd.DataFrame(rows).set_index("model")
cols = [f"overall_r@{k}", f"warm_r@{k}", f"cold_item_r@{k}", f"cold_user_r@{k}"]
display(tier2_df[cols])
tier2_df[cols].plot.bar(figsize=(10, 4), rot=0, title="Tier 2 four-view recall")
plt.ylabel(f"recall@{k}"); plt.legend(bbox_to_anchor=(1.02, 1)); plt.tight_layout()

## Next steps

- Headline run: `python scripts/benchmark.py --only lightgcn,image_embedding --force`
- Tier 3 modern methods: **`08_tier3_modern.ipynb`**